In [1]:
import pandas as pd

In [2]:
file_path = "/Users/lucasben/Documents/mba-business-analytics/Financial Performance Analysis/data/OMIS Store.xlsx"

orders = pd.read_excel(file_path, sheet_name=0)
returns = pd.read_excel(file_path, sheet_name=1)
customers = pd.read_excel(file_path, sheet_name=2)
employees = pd.read_excel(file_path, sheet_name=3)
products = pd.read_excel(file_path, sheet_name=4)

In [3]:
list_data = [orders, returns, customers, employees, products]

for df in list_data:
    print((df.isnull().sum() / len(df) * 100).round(2))

# 1.5% of product colour data is missing. I'll impute it with the most common colour for that product

Market              0.0
Region              0.0
Country             0.0
State               0.0
City                0.0
Employee ID         0.0
Customer ID         0.0
Order ID            0.0
Order Date          0.0
Year (OrderDate)    0.0
Order Priority      0.0
Product ID          0.0
Product Name        0.0
Category            0.0
Sub-Category        0.0
Segment             0.0
Ship Date           0.0
Ship Mode           0.0
Payment Type        0.0
Discount            0.0
Profit              0.0
Quantity            0.0
Sales               0.0
Shipping Cost       0.0
dtype: float64
Returned    0.0
Order ID    0.0
Market      0.0
dtype: float64
Customer ID       0.0
Customer Name     0.0
Sex               0.0
Market            0.0
Region            0.0
Rewards Member    0.0
dtype: float64
Employee ID       0.0
Employee Name     0.0
Market            0.0
Region            0.0
Title             0.0
Hire Date         0.0
Birth Date        0.0
Email Address     0.0
Marital Status    0.0
S

In [4]:
missing_colour = products[products['Colour'].isnull()]

missing_products = missing_colour['Product Name'].tolist()

missing_colour['Product Name'].unique()

array(['3M Office Air Cleaner', '3M Polarizing Light Filter Sleeves',
       "3M Replacement Filter for Office Air Cleaner for 20' x 33' Room",
       'American Pencil', 'Barrel Sharpener',
       'Berol Giant Pencil Sharpener', 'Binding Machine Supplies',
       'Blackstonian Pencils',
       'Boston 1645 Deluxe Heavier-Duty Electric Pencil Sharpener',
       'Boston 16701 Slimline Battery Pencil Sharpener',
       'Boston 16765 Mini Stand Up Battery Pencil Sharpener',
       'Boston 16801 Nautilus Battery Pencil Sharpener',
       'Boston 1730 StandUp Electric Pencil Sharpener',
       'Boston 1799 Powerhouse Electric Pencil Sharpener',
       'Boston 1827 Commercial Additional Cutter, Drive Gear & Gear Rack for 1606',
       'Boston 1900 Electric Pencil Sharpener',
       'Boston 19500 Mighty Mite Electric Pencil Sharpener',
       'Boston Heavy-Duty Trimline Electric Pencil Sharpeners',
       'Boston Home & Office Model 2000 Electric Pencil Sharpeners',
       'Boston KS Multi-Siz

In [5]:
product_colour_mode = products.groupby('Product Name')['Colour'].apply(lambda x: x.mode().iloc[0] if not x.mode().empty else 'Unknown').reset_index()

# .groupby('Product Name)['Colour] groups the data by unique product names and selects the colours from each group

# .apply(lambda x: ) applies a function to each group where x represents the colour series for each product group

# .iloc[0] if not x.mode().empty else 'Unknown' gest the first mode value and checks if there are any mode values, if no mode exists, returns  'Unknown'

# .reset_index() converts the result from a series into a dataframe

product_colour_mode.columns = ['Product Name', 'Mode_Colour']

print(product_colour_mode)

                                           Product Name Mode_Colour
0     "While you Were Out" Message Book, One Form pe...       Multi
1              #10 Gummed Flap White Envelopes, 100/Box       White
2                         #10 Self-Seal White Envelopes       White
3            #10 White Business Envelopes,4 1/8 x 9 1/2       White
4               #10- 4 1/8" x 9 1/2" Recycled Envelopes       White
...                                                 ...         ...
3792  iKross Bluetooth Portable Keyboard + Cell Phon...       White
3793                         iOttie HLCRIO102 Car Mount       Black
3794                                iOttie XL Car Mount       Black
3795  invisibleSHIELD by ZAGG Smudge-Free Screen Pro...       Clear
3796                 netTALK DUO VoIP Telephone Service       Black

[3797 rows x 2 columns]


In [6]:
products_imputed = products.merge(product_colour_mode, on='Product Name', how='left') # merging the mode data with the original data

products_imputed['Colour'] = products_imputed['Colour'].fillna(products_imputed['Mode_Colour']) # imputing missing product colours

products_imputed = products_imputed.drop('Mode_Colour', axis=1) # removing the temporary mode column

In [7]:
orders['Country'].nunique()

146

In [8]:
unemployment_data_canada = pd.read_csv('/Users/lucasben/Documents/mba-business-analytics/Financial Performance Analysis/data/unemployment_canada_data.csv')

In [9]:
unemployment_data_canada['LRUN64TTCAQ156S'].mean()

8.11423787437186

In [10]:
# how much business do we do with the US?

us_sales = orders[orders['Country'] == 'United States']

total_sales_us = us_sales['Sales'].sum()
total_sales_us

2301477.2699999996

In [11]:
total_sales = orders['Sales'].sum()
total_sales

12668676.879999999

In [12]:
total_sales_us / total_sales 

0.18166674324398743

In [13]:
ranking_by_sales = orders.groupby('Country')['Sales'].sum().sort_values(ascending=False)
ranking_by_sales

Country
United States        2301477.27
Australia             926314.27
France                861586.96
China                 701096.19
Germany               629819.55
                        ...    
Tajikistan               245.86
Macedonia                212.94
Eritrea                  189.96
Armenia                  157.97
Equatorial Guinea        149.97
Name: Sales, Length: 146, dtype: float64

In [14]:
obj = ranking_by_sales.iloc[1:4].sum()
obj

2488997.42

In [15]:
ranking_by_sales.iloc[0] - obj

-187520.1499999999

In [16]:
ranking_by_sales.iloc[0:11]

Country
United States     2301477.27
Australia          926314.27
France             861586.96
China              701096.19
Germany            629819.55
Mexico             624217.89
India              590324.42
United Kingdom     530762.39
Indonesia          405935.63
Brazil             362605.52
Italy              290989.74
Name: Sales, dtype: float64

In [17]:
emea_top_three = ranking_by_sales.iloc[[2,4,7]].sum()
emea_top_three

2022168.9

In [18]:
markets_by_sales = orders.groupby('Market')['Sales'].sum().sort_values(ascending=False)
markets_by_sales

Market
EMEA     4539191.46
APAC     3590434.74
USCA     2368518.94
LATAM    2170531.74
Name: Sales, dtype: float64

In [19]:
 4539191.46 - 2368518.94 - 2170531.74

140.7799999997951

In [20]:

    x = orders.groupby('Market')['Country'].count()
x


Market
APAC     11002
EMEA     19616
LATAM    10294
USCA     10378
Name: Country, dtype: int64

In [21]:
orders

,Market,Region,Country,State,City,Employee ID,Customer ID,Order ID,Order Date,Year (OrderDate),...,Sub-Category,Segment,Ship Date,Ship Mode,Payment Type,Discount,Profit,Quantity,Sales,Shipping Cost
0,EMEA,EMEA,Hungary,Budapest,Budapest,OMISSE00431,OMISS000299,HU-2021-626797,2021-01-01,2021,...,Storage,Consumer,2021-01-06,Express Shipping,Square,0.00,30.16,4,67.96,9.06
1,EMEA,Africa,Algeria,Constantine,Constantine,OMISSE00434,OMISS000561,DZ-2021-132562,2021-01-01,2021,...,Storage,Consumer,2021-01-07,Standard Shipping,Credit,0.00,106.83,2,422.98,35.65
2,APAC,Oceania,Australia,New South Wales,Wagga Wagga,OMISSE00059,OMISS000269,AU-2021-699976,2021-01-01,2021,...,Furnishings,Consumer,2021-01-09,Standard Shipping,Credit,0.12,38.87,5,111.95,4.90
3,APAC,Oceania,Australia,New South Wales,Wagga Wagga,OMISSE00059,OMISS000269,AU-2021-300725,2021-01-01,2021,...,Paper,Consumer,2021-01-09,Standard Shipping,Credit,0.11,15.68,2,55.98,2.52
4,APAC,Oceania,Australia,New South Wales,Wagga Wagga,OMISSE00059,OMISS000475,AU-2021-341993,2021-01-01,2021,...,Supplies,Consumer,2021-01-09,Standard Shipping,Credit,0.11,37.26,3,120.97,10.06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51285,LATAM,South,Brazil,São Paulo,São Paulo,OMISSE00696,OMISS000810,BR-2024-183663,2024-12-31,2024,...,Accessories,Corporate,2025-01-04,Express Shipping,Credit,0.00,29.49,1,74.99,7.26
51286,LATAM,South,Brazil,São Paulo,São Paulo,OMISSE00696,OMISS000810,BR-2024-432257,2024-12-31,2024,...,Copiers,Corporate,2025-01-04,Express Shipping,Apple Pay,0.00,302.38,5,1281.95,252.40
51287,EMEA,Africa,Morocco,Souss-Massa-Draâ,Agadir,OMISSE00434,OMISS000737,MA-2024-781569,2024-12-31,2024,...,Binders,Consumer,2025-01-06,Standard Shipping,Square,0.00,-0.04,1,4.99,0.60
51288,LATAM,North,Mexico,Tamaulipas,Reynosa,OMISSE00724,OMISS000516,MX-2024-261915,2024-12-31,2024,...,Labels,Home Office,2025-01-05,Standard Shipping,Apple Pay,0.00,0.41,3,16.97,1.41


In [22]:
orders['Sub-Category'].unique()

array(['Storage', 'Furnishings', 'Paper', 'Supplies', 'Machines',
       'Chairs', 'Accessories', 'Copiers', 'Bookcases', 'Phones',
       'Tables', 'Fasteners', 'Appliances', 'Labels', 'Envelopes', 'Art',
       'Binders'], dtype=object)

In [23]:
orders['Category'].unique()

array(['Office Supplies', 'Furniture', 'Technology'], dtype=object)

# **Analytics**

## **How much do we do in sales?**

In [24]:
orders['Sales'].sum() # total sales historically 

12668676.879999999

In [25]:
# sales delta from 2023 to 2024
(4310949.78 - 3412293.64) / 3412293.64 * 100

26.335838436225558

In [26]:
# sales delta from 2022 to 2023
(3412293.64 - 2682524.89) / 2682524.89 * 100

27.204547205524715

In [27]:
# sales delta from 2021 to 2022
(2682524.89 - 2262908.57) / 2262908.57 * 100

18.543229079732566

In [28]:
# average change in sales since 2021
(26.335838436225558 + 27.204547205524715 + 18.543229079732566) / 3

24.027871573827614

## **What are our sales in each market?**

In [29]:
# how many countries in each market

markets = orders['Market'].unique()

for market in sorted(markets):
    mask = orders['Market'] == market
    countries = orders.loc[mask, 'Country'].nunique()
    print(f"Market: {market}, Countries: {countries}")


Market: APAC, Countries: 23
Market: EMEA, Countries: 98
Market: LATAM, Countries: 24
Market: USCA, Countries: 2


In [30]:
# how many customers per market

for market in sorted(markets):
    mask = orders['Market'] == market
    customers = orders.loc[mask, 'Customer ID'].nunique()
    print(f"Market: {market}, Customers: {customers}")

Market: APAC, Customers: 217
Market: EMEA, Customers: 438
Market: LATAM, Customers: 177
Market: USCA, Customers: 213


In [31]:
customers['Customer ID'].nunique()

TypeError: 'int' object is not subscriptable

In [32]:
orders_2024 = orders[orders['Year (OrderDate)'] == 2024]

markets_2024 = orders_2024['Market'].unique()

for market in sorted(markets_2024):
    mask = orders_2024['Market'] == market
    customers_2024 = orders_2024.loc[mask, 'Customer ID'].nunique()
    print(f"Market {market}, Customers: {customers_2024}")

Market APAC, Customers: 217
Market EMEA, Customers: 438
Market LATAM, Customers: 177
Market USCA, Customers: 213


In [33]:
# 2024 tech sales delta compared to furniture 
(1619536 - 1379977) / 1379977 * 100

17.359637153372844

In [ ]:
# 2023 tech sales delta compared to furniture 
(1279045 - 1118770) / 1118770 * 100

14.326000875961995

## **In which countries do we do the most sales?**

In [ ]:
sales_by_country_2024 = orders[orders['Year (OrderDate)'] == 2024].groupby('Country')['Sales'].sum()

top_countries = sales_by_country_2024.sort_values(ascending=False)

for country, sales in top_countries.head(10).items():
    print(f"{country}: ${sales:,.2f}")

United States: $735,818.97
Australia: $314,492.42
France: $309,518.33
China: $218,766.07
Germany: $216,976.81
India: $205,498.65
Mexico: $196,363.67
United Kingdom: $194,575.41
Indonesia: $145,687.00
Brazil: $120,655.68


# **Our Current Situation**

### **Sales**

In [ ]:
years = orders['Year (OrderDate)'].unique() # getting all unique years from the data

# sales per year
for year in sorted(years):
    sales_year = orders[orders['Year (OrderDate)'] == year]
    total_sales = sales_year['Sales'].sum()
    print(f"For {year}, we made ${total_sales:,.2f} in sales")

For 2021, we made $2,262,908.57 in sales
For 2022, we made $2,682,524.89 in sales
For 2023, we made $3,412,293.64 in sales
For 2024, we made $4,310,949.78 in sales


In [ ]:
# our largest market is the US
us_sales_by_year = orders[orders['Country'] == 'United States'].groupby('Year (OrderDate)')['Sales'].sum()
print(us_sales_by_year)

Year (OrderDate)
2021    484747.19
2022    471386.21
2023    609524.90
2024    735818.97
Name: Sales, dtype: float64


In [34]:
import pandas as pd

# Assuming your DataFrame is named orders

# 1. Aggregate total sales by Country and Year
sales_by_year = (
    orders.groupby(['Country', 'Year (OrderDate)'])['Sales']
          .sum()
          .reset_index()
)

# 2. Pivot so each row is a country and each column is a year
sales_pivot = sales_by_year.pivot(index='Country', columns='Year (OrderDate)', values='Sales')

# 3. Filter to countries where 2024 < 2023
decreased_countries = sales_pivot[sales_pivot[2024] < sales_pivot[2023]]

decreased_countries


Year (OrderDate),2021,2022,2023,2024
Country,,,,
Albania,1725.85,959.92,819.94,415.89
Azerbaijan,802.92,479.93,2751.82,1645.79
Bolivia,2798.78,2521.71,4162.24,2118.53
Chile,2340.44,9935.99,15267.55,8046.51
Colombia,10563.27,12732.32,30326.08,28111.56
Democratic Republic of the Congo,16299.89,17452.38,29996.40,26564.00
Denmark,1215.72,939.76,3631.18,2866.14
Estonia,1976.96,1860.95,390.91,208.94
Ethiopia,430.91,NaN,252.95,173.96


In [35]:
# Extract unique country–market mapping
country_market = orders[['Country', 'Market']].drop_duplicates()
# Reset index so Country is a column
decreased_df = decreased_countries.reset_index()

# Merge to get the market for each country
decreased_with_market = decreased_df.merge(country_market, on='Country', how='left')

decreased_with_market


,Country,2021,2022,2023,2024,Market
0,Albania,1725.85,959.92,819.94,415.89,EMEA
1,Azerbaijan,802.92,479.93,2751.82,1645.79,EMEA
2,Bolivia,2798.78,2521.71,4162.24,2118.53,LATAM
3,Chile,2340.44,9935.99,15267.55,8046.51,LATAM
4,Colombia,10563.27,12732.32,30326.08,28111.56,LATAM
5,Democratic Republic of the Congo,16299.89,17452.38,29996.40,26564.00,EMEA
6,Denmark,1215.72,939.76,3631.18,2866.14,EMEA
7,Estonia,1976.96,1860.95,390.91,208.94,EMEA
8,Ethiopia,430.91,NaN,252.95,173.96,EMEA
9,Finland,4119.51,9354.13,6180.46,1132.89,EMEA


In [36]:
market_counts = decreased_with_market['Market'].value_counts()

market_counts


Market
EMEA     23
APAC      7
LATAM     6
Name: count, dtype: int64

In [37]:
summary = (
    decreased_with_market
    .groupby('Market')['Country']
    .nunique()
    .reset_index(name='Num_Countries_Decreased')
)

summary


,Market,Num_Countries_Decreased
0,APAC,7
1,EMEA,23
2,LATAM,6


In [38]:
# Reset index so Country is a column
decreased_df = decreased_countries.reset_index()

# Calculate percent change from 2023 → 2024
decreased_df['Percent_Change'] = ((decreased_df[2024] - decreased_df[2023]) / decreased_df[2023]) * 100


In [44]:
# Extract unique country–market mapping
country_market = orders[['Country', 'Market']].drop_duplicates()

# Merge country → market
decreased_with_market = decreased_df.merge(country_market, on='Country', how='left')

decreased_with_market.sort_values('Percent_Change')


,Country,2021,2022,2023,2024,Percent_Change,Market
26,Papua New Guinea,52.96,166.93,2549.52,150.83,-94.083984,APAC
32,Taiwan,6761.67,NaN,786.84,51.97,-93.395099,APAC
30,Sri Lanka,NaN,91.98,718.78,52.92,-92.637525,APAC
9,Finland,4119.51,9354.13,6180.46,1132.89,-81.669811,EMEA
27,Portugal,3292.38,1265.55,8693.95,1845.26,-78.775355,EMEA
33,Tunisia,25.98,521.93,985.94,224.93,-77.186239,EMEA
16,Jamaica,1624.75,1537.74,2676.56,930.83,-65.222898,LATAM
19,Lebanon,713.96,474.99,1105.92,505.95,-54.250760,EMEA
29,Singapore,8324.72,14024.48,12412.88,5814.38,-53.158493,APAC
22,Mali,54.99,2311.83,4334.63,2078.75,-52.043196,EMEA


In [41]:
avg_pct = (
    decreased_with_market
    .groupby('Market')['Percent_Change']
    .mean()
    .reset_index(name='Avg_Percent_Change')
)
avg_pct

,Market,Avg_Percent_Change
0,APAC,-50.742585
1,EMEA,-33.661272
2,LATAM,-34.835085
